# Rectified Limit State Space - Tests

Este notebook contiene pruebas y experimentos para el método de *rectified limit state space* con funciones de activación sigmoid y tanh, incluyendo:
- Relajación lineal (lower/upper relaxation)
- Iteración simbólica
- Propagación del espacio de estados
- Concretización de estados simbólicos

In [ ]:
import json


with open("maps.json", "r") as f:
    samples = json.loads(f.read())

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64
torch.set_grad_enabled(False)
torch.set_default_dtype(dtype)
torch.set_printoptions(precision=30, sci_mode=False)

print(f"Device: {device}")

In [ ]:
import core
from core import (
    generate_inputs,
    generate_input_samples,
    upwards_rounding,
    downwards_rounding,
    outwards_rounding,
    is_inside_state,
    state_series_from_samples,
    is_inside_state_series,
    margin,
    states_intersection,
    series_intersection,
    covering,
    average_gap,
    convergence_iteration,
)

In [ ]:
from core import (
    get_function_data,
    lower_relaxation,
    upper_relaxation,
    induced_state_space,
    concretize_state_space,
    propagate_state_space,
    apply_relaxed_activation,
    symbolic_iteration,
    naive_iteration,
    containment_guided_state_space,
    rectified_intersection_state_space,
    interval_state_space,
    rescaled_reasoning,
    overapproximation_from_relaxations,
    symbolic_state_space,
    warm_start_symbolic_state_space,
)

In [ ]:
from math import isnan
import pandas as pd
from tqdm import tqdm

import signal

interrupted = False


def handler(signum, frame):
    global interrupted
    interrupted = True


signal.signal(signal.SIGINT, handler)


# Setup
RESUME_TESTING = True
CSV_FILE = "resultados.csv"

# Testing constants and options
EPSILON = 1e-6
INPUT_LOWER_BOUND = 0.0
INPUT_UPPER_BOUND = 1.0
ACTIVATION_FUNCTION = "sigmoid"
NUM_SAMPLES = 100
NUM_ITERS = 50
PHI_OPTIONS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
SAVE_EVERY_ITERS = 20
MINIMAL_WARMING = 5

# Testing logic
data = []
dataframe = pd.DataFrame()
processed_sample_indexes = set()

if RESUME_TESTING:
    try:
        dataframe = pd.read_csv(CSV_FILE, index_col=0)

        processed_sample_indexes = set(zip(dataframe["uuid"], dataframe["phi"]))
    except FileNotFoundError:
        print("No se encontró el archivo .csv")

try:
    for sample_index, sample in enumerate(tqdm(samples)):
        if interrupted:
            print("Detenido por el usuario.")
            break

        n_concepts = sample["n_concepts"]

        inputs = generate_inputs(
            n_concepts=n_concepts,
            l=INPUT_LOWER_BOUND,
            u=INPUT_UPPER_BOUND,
            device=device,
        )
        W = torch.tensor(sample["weigths"], device=device)
        Wt = W.T

        for phi in PHI_OPTIONS:
            if interrupted:
                break

            if (sample["uuid"], phi) in processed_sample_indexes:
                continue

            stats = {
                "uuid": sample["uuid"],
                "n_concepts": n_concepts,
                "density": sample["density"],
                "phi": phi,
            }

            # Sampled Aproximation
            A = generate_input_samples(
                NUM_SAMPLES, n_concepts, INPUT_LOWER_BOUND, INPUT_UPPER_BOUND
            )
            simulated_states = rescaled_reasoning(
                A,
                W,
                T=NUM_ITERS,
                phi=phi,
                transfer_function=(
                    torch.sigmoid if ACTIVATION_FUNCTION == "sigmoid" else torch.tanh
                ),
                device=device,
            )
            sampled_states = state_series_from_samples(simulated_states)
            stats["sampled_covering"] = covering(sampled_states)

            # Naive Interval Analysis
            naive_states = interval_state_space(
                Wt, inputs, ACTIVATION_FUNCTION, phi, NUM_ITERS, device=device
            )
            stats["naive_covering"] = covering(naive_states)
            stats["naive_is_valid"] = is_inside_state_series(
                sampled_states, naive_states, tolerance=EPSILON
            )
            stats["naive_converged_at"] = convergence_iteration(naive_states, EPSILON)
            stats["naive_average_gap"] = average_gap(naive_states, simulated_states)

            # Symbolic Interval Analysis
            symbolic_states = symbolic_state_space(
                Wt,
                inputs,
                ACTIVATION_FUNCTION,
                phi,
                NUM_ITERS,
                overapprox_threshold=None,
                device=device,
            )
            symbolic_states = concretize_state_space(
                symbolic_states, inputs, device=device
            )

            stats["symbolic_covering"] = covering(symbolic_states)
            stats["symbolic_is_valid"] = is_inside_state_series(
                sampled_states, symbolic_states, tolerance=EPSILON
            )
            stats["symbolic_converged_at"] = convergence_iteration(
                symbolic_states, EPSILON
            )
            stats["symbolic_average_gap"] = average_gap(
                symbolic_states, simulated_states
            )

            naive_symbolic_intersection = series_intersection(
                naive_states, symbolic_states
            )

            # Rectified Interval Analysis
            rectified_states = rectified_intersection_state_space(
                Wt,
                inputs,
                ACTIVATION_FUNCTION,
                phi,
                NUM_ITERS,
                minimal_warming_iters=MINIMAL_WARMING,
                baseline_interval_series=naive_symbolic_intersection,
            )
            stats["rectified_covering"] = covering(rectified_states)
            stats["rectified_is_valid"] = is_inside_state_series(
                sampled_states, rectified_states, tolerance=EPSILON
            )
            stats["rectified_converged_at"] = convergence_iteration(
                rectified_states, EPSILON
            )
            stats["rectified_average_gap"] = average_gap(
                rectified_states, simulated_states
            )

            stats["rectified_is_inside_naive"] = is_inside_state_series(
                rectified_states, naive_states, tolerance=EPSILON
            )
            stats["rectified_is_inside_symbolic"] = is_inside_state_series(
                rectified_states, symbolic_states, tolerance=EPSILON
            )

            stats["margin"] = margin(rectified_states, symbolic_states)

            if (
                not stats["rectified_is_valid"]
                or not stats["naive_is_valid"]
                or not stats["symbolic_is_valid"]
            ):
                print("No valid value encountered!!!")

            if (
                not stats["rectified_is_inside_naive"]
                or not stats["rectified_is_inside_symbolic"]
            ):
                print("Rectified is not inside Naive or Symbolic!!!")

            if any([isnan(x) for x in stats.values() if isinstance(x, float)]):
                print("NaN values found!!!")

            data.append(stats)

        if (sample_index + 1) % SAVE_EVERY_ITERS == 0:
            dataframe = pd.concat([dataframe, pd.DataFrame(data)], ignore_index=True)
            dataframe.to_csv(CSV_FILE)
            data = []
finally:
    dataframe = pd.concat([dataframe, pd.DataFrame(data)], ignore_index=True)
    dataframe.to_csv(CSV_FILE)

In [ ]:
print("NaN Values Report Per Column:")
print(dataframe.isnull().sum())
print()

In [ ]:
df_nans = dataframe[
    dataframe[
        [
            "naive_covering",
            "naive_average_gap",
            "symbolic_covering",
            "symbolic_average_gap",
            "rectified_covering",
            "rectified_average_gap",
        ]
    ]
    .isnull()
    .any(axis=1)
]
df_nans

In [ ]:
df_inside_naive_false = dataframe[
    (dataframe[["rectified_is_inside_naive"]] == False).any(axis=1)
]
df_inside_naive_false

In [ ]:
df_inside_symbolic_false = dataframe[
    (dataframe[["rectified_is_valid"]] == False).any(axis=1)
]
df_inside_symbolic_false

In [ ]:
df_rectified_is_valid_false = dataframe[
    (dataframe[["rectified_is_inside_symbolic"]] == False).any(axis=1)
]
df_rectified_is_valid_false

### De aquí debemos mostrar que:

- los intervalos encontrados en cada iteración son siempre más reducidos (o iguales) que los de los métodos aritmético y simbólico por sí solos. tmb q contienen a los samples

- el cubrimiento es siempre menor en el método propuesto (obvio ... se deriva de lo anterior)

- resaltar que el método simbólico sobreaproxima cuando phi se acerca a 1 y aumenta la cantidad de neuronas. Ahí el método propuesto saca ventaja clara.


### Gráficas:

- hacer un gráfico que incluya tres valores de phi, tres valores de # de neuronas, y tres valores de conectividad. Para cada combinación tener una figura, y mostrar en colores las variantes aritmética, simbólica y nuestra propuesta, con los min y max. El objetivo es ver q la propuesta siempre está contenida en las demás, aunque mostremos un solo ejemplo.

- hacer gráficos mostrando el cubrimiento promedio para las tres variantes de algoritmos. Uno con phi en el eje X,  otro con el número de neuronas, y otro con la conectividad.

- sería cool mostrar el promedio de la desviación estándar de cada método a los samples. variando phi

### Covering

In [ ]:
# Mapeo para mostrar nombres amigables en la leyenda
method_names = {
    "naive": "Aritmético",
    "rectified": "Rectificación",
    "symbolic": "Simbólico",
    "sampled": "Activaciones Reales",
}


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math

# ===============================
# --- Cargar datos ---
# ===============================
df = pd.read_csv("resultados.csv")

# ===============================
# --- Reestructurar (melt) ---
# ===============================
df_melted = df.melt(
    id_vars=["uuid", "n_concepts", "phi", "density"],
    value_vars=[
        "naive_covering",
        "rectified_covering",
        "sampled_covering",
        "symbolic_covering",
    ],
    var_name="covering_type",
    value_name="value",
)

# ===============================
# --- Normalizar nombres ---
# ===============================
df_melted["covering_type"] = df_melted["covering_type"].str.replace("_covering", "")

# ===============================
# --- Asegurar tipos numéricos ---
# ===============================
for col in ["phi", "value", "density"]:
    df_melted[col] = pd.to_numeric(df_melted[col], errors="coerce")

# ===============================
# --- Calcular estadísticas ---
# ===============================
stats = (
    df_melted.groupby(["n_concepts", "density", "phi", "covering_type"], dropna=False)[
        "value"
    ]
    .agg(
        mean="mean",
        min="min",
        max="max",
    )
    .reset_index()
)

sns.set(style="whitegrid")

# ===============================
# --- Valores deseados ---
# ===============================
selected_n_concepts = [10, 15, 20]  # reemplaza con tus valores
selected_density = [0.4, 0.6, 0.9]  # reemplaza con tus valores

# ===============================
# --- Filtrar datos ---
# ===============================
stats_filtered = stats[
    stats["n_concepts"].isin(selected_n_concepts)
    & stats["density"].isin(selected_density)
]

# ===============================
# --- Configuración de Subplots ---
# ===============================
rows = len(selected_n_concepts)
cols = len(selected_density)

fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), squeeze=False)

# ===============================
# --- Colores personalizados ---
# ===============================
colors = {
    "naive": "#1f77b4",  # azul
    "rectified": "red",  # rojo
    "symbolic": "#4B0082",  # violeta oscuro
    "sampled": "black",  # negro
}

# ===============================
# --- Dibujar gráficos ---
# ===============================
for i, n_concepts in enumerate(selected_n_concepts):
    for j, density_val in enumerate(selected_density):
        ax = axes[i, j]
        df_sub = stats_filtered[
            (stats_filtered["n_concepts"] == n_concepts)
            & (stats_filtered["density"] == density_val)
        ]

        if df_sub.empty:
            ax.axis("off")
            continue

        for base_type in ["naive", "rectified", "sampled", "symbolic"]:
            df_c = df_sub[df_sub["covering_type"] == base_type]
            if not df_c.empty:
                x = df_c["phi"]
                y = df_c["mean"]
                y_min = df_c["min"]
                y_max = df_c["max"]
                (line,) = ax.plot(
                    x,
                    y,
                    label=method_names.get(base_type, base_type),
                    linewidth=2,
                    color=colors[base_type],
                )
                ax.fill_between(x, y_min, y_max, alpha=0.25, color=colors[base_type])

        ax.set_title(f"Num. Conceptos={n_concepts}  Densidad={density_val}")
        ax.set_xlabel("φ")
        ax.set_ylabel("Cubrimiento")
        ax.legend(loc="lower left", fontsize="small")

# fig.suptitle("Análisis Min-Max por N Concepts y Density", fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.show()

### Gap

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ===============================
# --- Cargar datos ---
# ===============================
df = pd.read_csv("resultados.csv")

# ===============================
# --- Reestructurar (melt) ---
# ===============================
df_melted = df.melt(
    id_vars=["uuid", "n_concepts", "phi", "density"],
    value_vars=[
        "naive_average_gap",
        "rectified_average_gap",
        "symbolic_average_gap",
    ],
    var_name="method",
    value_name="gap",
)

# ===============================
# --- Normalizar nombres ---
# ===============================
df_melted["method"] = df_melted["method"].str.replace("_average_gap", "")

# ===============================
# --- Asegurar tipos numéricos ---
# ===============================
for col in ["phi", "gap", "density"]:
    df_melted[col] = pd.to_numeric(df_melted[col], errors="coerce")

# ===============================
# --- Calcular estadísticas min–max ---
# ===============================
stats = (
    df_melted.groupby(["n_concepts", "density", "phi", "method"], dropna=False)["gap"]
    .agg(
        mean="mean",
        min="min",
        max="max",
    )
    .reset_index()
)

sns.set(style="whitegrid")

# ===============================
# --- Valores deseados para subplots ---
# ===============================
selected_n_concepts = [10, 15, 20]  # reemplaza con tus valores
selected_density = [0.4, 0.6, 0.9]  # reemplaza con tus valores

stats_filtered = stats[
    stats["n_concepts"].isin(selected_n_concepts)
    & stats["density"].isin(selected_density)
]

# ===============================
# --- Configuración de Subplots ---
# ===============================
rows = len(selected_n_concepts)
cols = len(selected_density)

fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), squeeze=False)

# ===============================
# --- Colores igual que el gráfico de coverings ---
# ===============================
colors = {
    "naive": "#1f77b4",  # azul (como Seaborn default)
    "rectified": "red",  # naranja
    "symbolic": "#4B0082",  # verde
}


# ===============================
# --- Dibujar gráficos ---
# ===============================
for i, n_concepts in enumerate(selected_n_concepts):
    for j, density_val in enumerate(selected_density):
        ax = axes[i, j]
        df_sub = stats_filtered[
            (stats_filtered["n_concepts"] == n_concepts)
            & (stats_filtered["density"] == density_val)
        ]

        if df_sub.empty:
            ax.axis("off")
            continue

        for method in ["naive", "rectified", "symbolic"]:
            df_m = df_sub[df_sub["method"] == method]
            if not df_m.empty:
                x = df_m["phi"]
                y = df_m["mean"]
                y_min = df_m["min"]
                y_max = df_m["max"]
                (line,) = ax.plot(
                    x,
                    y,
                    label=method_names.get(method, method),
                    linewidth=2,
                    color=colors[method],
                )
                ax.fill_between(x, y_min, y_max, alpha=0.25, color=colors[method])

        ax.set_title(f"Num. Conceptos={n_concepts}  Densidad={density_val}")
        ax.set_xlabel("φ")
        ax.set_ylabel("Desviación")
        ax.legend(loc="upper left", fontsize="small")

# fig.suptitle("Análisis Min–Max de Average Gaps por N Concepts y Density", fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.show()

### n_concepts en el eje x

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ===============================
# --- Cargar datos ---
# ===============================
df = pd.read_csv("resultados.csv")

# ===============================
# --- Reestructurar (melt) ---
# ===============================
df_melted = df.melt(
    id_vars=["uuid", "n_concepts", "phi", "density"],
    value_vars=[
        "naive_covering",
        "rectified_covering",
        "sampled_covering",
        "symbolic_covering",
    ],
    var_name="covering_type",
    value_name="value",
)

# ===============================
# --- Normalizar nombres ---
# ===============================
df_melted["covering_type"] = df_melted["covering_type"].str.replace("_covering", "")

# ===============================
# --- Asegurar tipos numéricos ---
# ===============================
for col in ["phi", "value", "density", "n_concepts"]:
    df_melted[col] = pd.to_numeric(df_melted[col], errors="coerce")

# ===============================
# --- Calcular estadísticas min–max ---
# ===============================
stats = (
    df_melted.groupby(["n_concepts", "density", "phi", "covering_type"], dropna=False)[
        "value"
    ]
    .agg(mean="mean", min="min", max="max")
    .reset_index()
)

sns.set(style="whitegrid")

# ===============================
# --- Valores deseados ---
# ===============================
selected_phi = [0.4, 0.6, 0.9]  # valores de phi
selected_density = [0.4, 0.6, 0.9]  # valores de density
n_concepts_range = list(range(2, 21))  # eje x

# ===============================
# --- Filtrar datos ---
# ===============================
stats_filtered = stats[
    stats["phi"].isin(selected_phi)
    & stats["density"].isin(selected_density)
    & stats["n_concepts"].isin(n_concepts_range)
]

# ===============================
# --- Configuración de Subplots ---
# ===============================
rows = len(selected_phi)
cols = len(selected_density)

fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), squeeze=False)

# ===============================
# --- Colores personalizados ---
# ===============================
colors = {
    "naive": "#1f77b4",  # azul
    "rectified": "red",  # rojo
    "symbolic": "#4B0082",  # violeta oscuro
    "sampled": "black",  # negro
}

# ===============================
# --- Dibujar gráficos ---
# ===============================
for i, phi_val in enumerate(selected_phi):
    for j, density_val in enumerate(selected_density):
        ax = axes[i, j]
        df_sub = stats_filtered[
            (stats_filtered["phi"] == phi_val)
            & (stats_filtered["density"] == density_val)
        ]

        if df_sub.empty:
            ax.axis("off")
            continue

        for method in ["naive", "rectified", "sampled", "symbolic"]:
            df_m = df_sub[df_sub["covering_type"] == method]
            if not df_m.empty:
                x = df_m["n_concepts"]
                y = df_m["mean"]
                y_min = df_m["min"]
                y_max = df_m["max"]
                (line,) = ax.plot(x, y, label=method, linewidth=2, color=colors[method])
                ax.fill_between(x, y_min, y_max, alpha=0.25, color=colors[method])

        ax.set_title(f"phi={phi_val}, density={density_val}")
        ax.set_xlabel("n_concepts")
        ax.set_ylabel("covering")
        ax.set_xticks(n_concepts_range)
        ax.legend(loc="lower left", fontsize="small")

fig.suptitle("Análisis Min-Max por N Concepts filtrado por Phi y Density", fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math

# ===============================
# --- Cargar datos ---
# ===============================
df = pd.read_csv("resultados.csv")

# ===============================
# --- Reestructurar (melt) ---
# ===============================
df_melted = df.melt(
    id_vars=["uuid", "n_concepts", "phi", "density"],
    value_vars=[
        "naive_covering",
        "rectified_covering",
        "sampled_covering",
        "symbolic_covering",
    ],
    var_name="covering_type",
    value_name="value",
)

# ===============================
# --- Normalizar nombres ---
# ===============================
df_melted["covering_type"] = df_melted["covering_type"].str.replace("_covering", "")

# ===============================
# --- Asegurar tipos numéricos ---
# ===============================
for col in ["phi", "value", "density"]:
    df_melted[col] = pd.to_numeric(df_melted[col], errors="coerce")

# ===============================
# --- Calcular estadísticas min–max ---
# ===============================
stats = (
    df_melted.groupby(["n_concepts", "density", "phi", "covering_type"], dropna=False)[
        "value"
    ]
    .agg(
        mean="mean",
        min="min",
        max="max",
    )
    .reset_index()
)

sns.set(style="whitegrid")

# ===============================
# --- Valores deseados para filtrado ---
# ===============================
selected_n_concepts = [10, 15, 20]  # reemplaza con tus valores
selected_phi = [0.4, 0.6, 0.9]  # nuevos valores de phi a filtrar

stats_filtered = stats[
    stats["n_concepts"].isin(selected_n_concepts) & stats["phi"].isin(selected_phi)
]

# ===============================
# --- Configuración de Subplots ---
# ===============================
rows = len(selected_n_concepts)
cols = len(selected_phi)

fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), squeeze=False)

# ===============================
# --- Colores personalizados ---
# ===============================
colors = {
    "naive": "#1f77b4",  # azul
    "rectified": "red",  # rojo
    "symbolic": "#4B0082",  # violeta oscuro
    "sampled": "black",  # negro
}

# # Mapeo para mostrar nombres amigables en la leyenda
# method_names = {
#     "naive": "Arithmetic",
#     "rectified": "Rectified",
#     "symbolic": "Symbolic",
#     "sampled": "Sampled",
# }

# ===============================
# --- Dibujar gráficos ---
# ===============================
for i, n_concepts in enumerate(selected_n_concepts):
    for j, phi_val in enumerate(selected_phi):
        ax = axes[i, j]
        df_sub = stats_filtered[
            (stats_filtered["n_concepts"] == n_concepts)
            & (stats_filtered["phi"] == phi_val)
        ]

        if df_sub.empty:
            ax.axis("off")
            continue

        for base_type in ["naive", "rectified", "sampled", "symbolic"]:
            df_c = df_sub[df_sub["covering_type"] == base_type]
            if not df_c.empty:
                # ordenar por density para líneas coherentes
                df_c = df_c.sort_values("density")
                x = df_c["density"]  # eje x ahora es density
                y = df_c["mean"]
                y_min = df_c["min"]
                y_max = df_c["max"]
                # usar el nombre amigable en la leyenda
                (line,) = ax.plot(
                    x,
                    y,
                    label=method_names.get(base_type, base_type),
                    linewidth=2,
                    color=colors[base_type],
                )
                ax.fill_between(x, y_min, y_max, alpha=0.25, color=colors[base_type])

        ax.set_title(f"n_concepts={n_concepts}, phi={phi_val}")
        ax.set_xlabel("density")
        ax.set_ylabel("covering")
        ax.legend(loc="upper left", fontsize="small")

fig.suptitle("Análisis Min-Max por N Concepts y Phi", fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # Necesario para 3D
from matplotlib import cm

# ===============================
# --- Cargar datos ---
# ===============================
df = pd.read_csv("resultados.csv")

# ===============================
# --- Normalizar nombres y tipos ---
# ===============================
df_melted = df.melt(
    id_vars=["uuid", "n_concepts", "phi", "density"],
    value_vars=["naive_average_gap", "rectified_average_gap", "symbolic_average_gap"],
    var_name="method",
    value_name="gap",
)
df_melted["method"] = df_melted["method"].str.replace("_average_gap", "")
for col in ["n_concepts", "density", "gap"]:
    df_melted[col] = pd.to_numeric(df_melted[col], errors="coerce")

# ===============================
# --- Filtrar método (ejemplo: rectified) ---
# ===============================
method = "rectified"
df_plot = df_melted[df_melted["method"] == method]

# ===============================
# --- Promediar gaps por n_concepts y density ---
# ===============================
pivot = df_plot.groupby(["n_concepts", "density"])["gap"].mean().reset_index()

# Crear mallas para superficie
X = pivot["n_concepts"].values
Y = pivot["density"].values
Z = pivot["gap"].values

# Rehacer como grid 2D para surface plot
n_concepts_unique = np.sort(pivot["n_concepts"].unique())
density_unique = np.sort(pivot["density"].unique())
X_grid, Y_grid = np.meshgrid(n_concepts_unique, density_unique)

Z_grid = np.empty_like(X_grid, dtype=float)

# Rellenar Z_grid
for i, d in enumerate(density_unique):
    for j, n in enumerate(n_concepts_unique):
        val = pivot[(pivot["n_concepts"] == n) & (pivot["density"] == d)]["gap"]
        Z_grid[i, j] = val.values[0] if not val.empty else np.nan

# ===============================
# --- Graficar superficie 3D ---
# ===============================
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

# Superficie con colormap de azul a rojo
surf = ax.plot_surface(
    X_grid,
    Y_grid,
    Z_grid,
    cmap=cm.get_cmap("coolwarm"),
    edgecolor="k",
    linewidth=0.2,
    alpha=0.9,
)

ax.set_xlabel("Número de Conceptos")
ax.set_ylabel("Densidad")
ax.set_zlabel("Gap")
ax.set_title(f"Surface plot del Gap ({method})")

# fig.colorbar(surf, shrink=0.5, aspect=10, label="Gap")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

# ===============================
# --- Cargar datos ---
# ===============================
df = pd.read_csv("resultados.csv")

# ===============================
# --- Normalizar nombres y tipos ---
# ===============================
df_melted_gap = df.melt(
    id_vars=["uuid", "n_concepts", "phi", "density"],
    value_vars=["naive_average_gap", "rectified_average_gap", "symbolic_average_gap"],
    var_name="method",
    value_name="gap",
)
df_melted_gap["method"] = df_melted_gap["method"].str.replace("_average_gap", "")
for col in ["n_concepts", "density", "gap"]:
    df_melted_gap[col] = pd.to_numeric(df_melted_gap[col], errors="coerce")

df_melted_covering = df.melt(
    id_vars=["uuid", "n_concepts", "phi", "density"],
    value_vars=["naive_covering", "rectified_covering", "symbolic_covering"],
    var_name="method",
    value_name="covering",
)
df_melted_covering["method"] = df_melted_covering["method"].str.replace("_covering", "")
for col in ["n_concepts", "density", "covering"]:
    df_melted_covering[col] = pd.to_numeric(df_melted_covering[col], errors="coerce")

# ===============================
# --- Filtrar método (ejemplo: rectified) ---
# ===============================
method = "rectified"
df_gap = df_melted_gap[df_melted_gap["method"] == method]
df_covering = df_melted_covering[df_melted_covering["method"] == method]

# ===============================
# --- Promediar por n_concepts y density ---
# ===============================
pivot_gap = df_gap.groupby(["n_concepts", "density"])["gap"].mean().reset_index()
pivot_covering = (
    df_covering.groupby(["n_concepts", "density"])["covering"].mean().reset_index()
)

# Crear grid 2D
n_concepts_unique = np.sort(df["n_concepts"].unique())
density_unique = np.sort(df["density"].unique())
X_grid, Y_grid = np.meshgrid(n_concepts_unique, density_unique)


def create_Z_grid(pivot, value_col):
    Z_grid = np.empty_like(X_grid, dtype=float)
    for i, d in enumerate(density_unique):
        for j, n in enumerate(n_concepts_unique):
            val = pivot[(pivot["n_concepts"] == n) & (pivot["density"] == d)][value_col]
            Z_grid[i, j] = val.values[0] if not val.empty else np.nan
    return Z_grid


Z_gap = create_Z_grid(pivot_gap, "gap")
Z_covering = create_Z_grid(pivot_covering, "covering")

# ===============================
# --- Graficar 2 subplots 3D ---
# ===============================
fig = plt.figure(figsize=(18, 7))

# Gap
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
surf1 = ax1.plot_surface(
    X_grid, Y_grid, Z_gap, cmap=cm.coolwarm, edgecolor="k", linewidth=0.2, alpha=0.9
)
ax1.set_xlabel("Número de Conceptos")
ax1.set_ylabel("Densidad")
ax1.set_zlabel("Gap")
ax1.set_title(f"Gap ({method})")
fig.colorbar(surf1, ax=ax1, shrink=0.5, aspect=10, label="Gap")

# Covering
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
surf2 = ax2.plot_surface(
    X_grid, Y_grid, Z_covering, cmap=cm.viridis, edgecolor="k", linewidth=0.2, alpha=0.9
)
ax2.set_xlabel("Número de Conceptos")
ax2.set_ylabel("Densidad")
ax2.set_zlabel("Covering")
ax2.set_title(f"Covering ({method})")
fig.colorbar(surf2, ax=ax2, shrink=0.5, aspect=10, label="Covering")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

# ===============================
# --- Cargar datos ---
# ===============================
df = pd.read_csv("resultados.csv")

# ===============================
# --- Normalizar nombres y tipos ---
# ===============================
df_melted = df.melt(
    id_vars=["uuid", "n_concepts", "phi", "density"],
    value_vars=["naive_average_gap", "rectified_average_gap", "symbolic_average_gap"],
    var_name="method",
    value_name="gap",
)
df_melted["method"] = df_melted["method"].str.replace("_average_gap", "")
for col in ["n_concepts", "density", "gap"]:
    df_melted[col] = pd.to_numeric(df_melted[col], errors="coerce")

# ===============================
# --- Configurar métodos y subplots ---
# ===============================
methods = ["naive", "symbolic", "rectified"]
fig = plt.figure(figsize=(18, 5))

method_names = {
    "naive": "Aritmético",
    "symbolic": "Simbólico",
    "rectified": "Rectificación",
}

for idx, method in enumerate(methods, 1):
    df_plot = df_melted[df_melted["method"] == method]

    # Promediar gaps por n_concepts y density
    pivot = df_plot.groupby(["n_concepts", "density"])["gap"].mean().reset_index()

    # Crear grids para superficie
    n_concepts_unique = np.sort(pivot["n_concepts"].unique())
    density_unique = np.sort(pivot["density"].unique())
    X_grid, Y_grid = np.meshgrid(n_concepts_unique, density_unique)
    Z_grid = np.empty_like(X_grid, dtype=float)

    for i, d in enumerate(density_unique):
        for j, n in enumerate(n_concepts_unique):
            val = pivot[(pivot["n_concepts"] == n) & (pivot["density"] == d)]["gap"]
            Z_grid[i, j] = val.values[0] if not val.empty else np.nan

    # Subplot 3D
    ax = fig.add_subplot(1, 3, idx, projection="3d")
    surf = ax.plot_surface(
        X_grid,
        Y_grid,
        Z_grid,
        cmap=cm.get_cmap("coolwarm"),
        edgecolor="k",
        linewidth=0.2,
        alpha=0.9,
    )
    ax.set_xlabel("Número de Conceptos")
    ax.set_ylabel("Densidad")
    ax.set_zlabel("Desviación")
    ax.set_title(f"{method_names[method]}")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

# ===============================
# --- Cargar datos ---
# ===============================
df = pd.read_csv("resultados.csv")

# ===============================
# --- Normalizar nombres y tipos ---
# ===============================
df_melted = df.melt(
    id_vars=["uuid", "n_concepts", "phi", "density"],
    value_vars=["naive_covering", "symbolic_covering", "rectified_covering"],
    var_name="method",
    value_name="covering",
)
df_melted["method"] = df_melted["method"].str.replace("_covering", "")
for col in ["n_concepts", "density", "covering"]:
    df_melted[col] = pd.to_numeric(df_melted[col], errors="coerce")

# ===============================
# --- Configurar métodos y subplots ---
# ===============================
methods = ["naive", "symbolic", "rectified"]
method_names = {
    "naive": "Aritmético",
    "symbolic": "Simbólico",
    "rectified": "Rectificación",
}

fig = plt.figure(figsize=(18, 5))

for idx, method in enumerate(methods, 1):
    df_plot = df_melted[df_melted["method"] == method]

    # Promediar covering por n_concepts y density
    pivot = df_plot.groupby(["n_concepts", "density"])["covering"].mean().reset_index()

    # Crear grids para superficie
    n_concepts_unique = np.sort(pivot["n_concepts"].unique())
    density_unique = np.sort(pivot["density"].unique())
    X_grid, Y_grid = np.meshgrid(n_concepts_unique, density_unique)
    Z_grid = np.empty_like(X_grid, dtype=float)

    for i, d in enumerate(density_unique):
        for j, n in enumerate(n_concepts_unique):
            val = pivot[(pivot["n_concepts"] == n) & (pivot["density"] == d)][
                "covering"
            ]
            Z_grid[i, j] = val.values[0] if not val.empty else np.nan

    # Subplot 3D
    ax = fig.add_subplot(1, 3, idx, projection="3d")
    surf = ax.plot_surface(
        X_grid,
        Y_grid,
        Z_grid,
        cmap=cm.get_cmap("coolwarm"),
        edgecolor="k",
        linewidth=0.2,
        alpha=0.9,
    )
    ax.set_xlabel("Número de Conceptos")
    ax.set_ylabel("Densidad")
    ax.set_zlabel("Cubrimiento")
    ax.set_title(f"{method_names[method]}")  # <- Aquí se usa el nombre custom

plt.tight_layout()
plt.show()